# 记忆压缩（Memory Compaction）

> **通过渐进式摘要（Progressive Summarization）降低存储与 token 成本：从原始记录到要点、到一句话摘要、再到标签。**

想象你把工作中的每场会议都逐字记录下来。一年后你会积累几千页内容，其中大部分是填充语：「我查一下」「好，没问题」「抱歉，你刚说什么？」真正的决策和行动项大概只占全文的 10%。

再想象你为每场会议保留四个版本：完整记录、要点列表、一句话摘要、一组标签。快速检索时你扫一眼标签；需要上下文时你读一句话摘要；只有在措辞本身很重要时，你才打开完整记录。

记忆压缩把这一思路用在智能体的记忆库上。原始记忆（完整对话记录、详细的工具输出、冗长的推理过程）会消耗大量存储和 token 预算。token 预算指把文本放进提示词的成本。压缩把冗长的记录转换成更紧凑的版本，保留关键信息，丢弃填充内容。

这项技术借鉴 Tiago Forte 为个人知识管理提出的渐进式摘要方法。每条记忆同时存在于多个保真度（fidelity）级别：级别 0（L0）是原始记录；级别 1（L1）提取要点；级别 2（L2）把它蒸馏成一句话；级别 3（L3）把它缩减为标签和实体引用（人名、日期、项目名这类结构化标签）。

实际收益相当可观。在同样的存储预算下，压缩后的记忆库能容纳 10 到 100 倍的语义内容。条目变小，检索也随之加快。智能体还能按场景选择合适的详细程度。

**读完本 notebook 你将理解：**
- 如何通过四个渐进级别压缩记忆。
- 如何构建一个每级调用 LLM 的压缩引擎。
- 如何根据存留时间和访问模式安排压缩。
- 如何按需检索任意保真度级别的记忆。


## 核心概念

- **渐进式摘要级别**：一套分层的压缩体系。L0 是原始文本，L1 是要点，L2 是一句话摘要，L3 只剩标签和实体引用。每升一级，都是用保真度（完整性）换取紧凑度（更小体积）。
- **有损与无损压缩**：无损技术（比如从非结构化文本中抽取结构化字段）保留全部信息；有损技术（比如摘要）有意丢弃细节。压缩会策略性地同时使用两者。
- **按级别检索**：在任意压缩级别上取回记忆的能力。快速浏览用 L2 或 L3，深入追查请求 L0 或 L1。这样智能体可以精确分配上下文 token。
- **压缩调度**：决定何时触发压缩的策略。可选方案包括按存留时间（N 小时后压缩）、按访问情况（M 个轮次未被访问后压缩）和按存储压力（存储超出容量时压缩）。
- **可逆性**：把原始 L0 保留在冷存储（便宜但较慢的存储）中，压缩因此不具破坏性。一旦摘要不够用，智能体随时可以「放大」回原始记忆。
- **存储与保真度的权衡**：决定每一级保留多少细节的工程决策。你是在存储成本和信息损失之间寻找平衡。
- **实体抽取**：作为压缩流程的一部分，从原始文本中抽取结构化实体（人物、日期、项目、决策）。它在压缩记忆之外，额外生成一份可查询的索引。


## 架构

<p align="center">
  <img src="../../images/diagrams/15_memory_compaction.svg" alt="记忆压缩示意图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    RM["Raw Memory\n(L0: Full fidelity)"] --> CE["Compaction\nEngine"]

    CE --> L1["L1: Key Points\n(~30% of original)"]
    L1 --> CE2["Compaction\nEngine"]
    CE2 --> L2["L2: One-liner\n(~5% of original)"]
    L2 --> CE3["Compaction\nEngine"]
    CE3 --> L3["L3: Tags Only\n(~1% of original)"]

    RM -->|"Archive original"| CS["Cold Storage"]

    RQ["Retrieval Query"] --> LS["Level Selector"]
    LS -->|"Quick scan"| L3
    LS -->|"Summary"| L2
    LS -->|"Moderate detail"| L1
    LS -->|"Full detail"| CS

    style RM fill:#2d4a7a,stroke:#49a,color:#fff
    style L1 fill:#2d5a5a,stroke:#4aa,color:#fff
    style L2 fill:#5a5a2d,stroke:#aa4,color:#fff
    style L3 fill:#5a2d2d,stroke:#a44,color:#fff
```

</details>

**数据流**：原始记忆（L0）以完整保真度进入系统，压缩引擎通过三个阶段逐步压缩它。L1 提取要点（约为原始 token 的 30%），L2 蒸馏成单句摘要（约 5%），L3 只保留标签和实体引用（约 1%）。

原始 L0 会归档进冷存储，因此你随时可以找回完整细节。检索查询到达时，级别选择器会挑选合适的详细级别，它同时考虑查询需求和可用的上下文预算。

## 环境准备

安装依赖并配置 API 访问。你需要一个 OpenAI API key，放在 `.env` 文件或环境变量里。


In [ ]:
%pip install -q python-dotenv deepagents langchain-openai

导入 OpenAI SDK 和标准库工具。API key 从 `.env` 文件加载。


In [ ]:
import os
import json
import time
import uuid
import re
from enum import IntEnum
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们要构建四个组件：

1. **CompactionLevel** 与 **CompactedMemory**：在多个保真度级别上保存内容的数据结构。
2. **CompactionEngine**：通过 LLM 调用完成从上一级到下一级的压缩。
3. **MemoryStore**：存储记忆，支持按级别检索。
4. **CompactionScheduler**：决定哪些记忆需要继续压缩。


### 数据结构

把它想象成一个有四个抽屉的档案柜。最上面的抽屉放完整文档，第二个放夹在每份文档上的要点摘录，第三个放便利贴写的梗概，最下面的抽屉只放写了关键词的索引卡。每个抽屉代表一种不同的「速度换细节」取舍。

`CompactionLevel` 是一个整数枚举，给这四个抽屉编号。`CompactedMemory` 保存目前已算好的那些级别的内容。


In [ ]:
class CompactionLevel(IntEnum):
    """四个渐进压缩级别。"""
    L0_RAW = 0          # 完整原始文本
    L1_KEY_POINTS = 1   # 要点提取（约为 L0 的 30%）
    L2_SUMMARY = 2      # 单句蒸馏（约为 L0 的 5%）
    L3_TAGS = 3         # 仅保留标签和实体（约为 L0 的 1%）


@dataclass
class CompactedMemory:
    """以一个或多个压缩级别存储的记忆。"""
    memory_id: str
    levels: dict          # CompactionLevel -> 字符串内容
    entities: list        # 抽取出的实体名称
    created_at: float     # Unix 时间戳
    last_accessed: float  # Unix 时间戳
    access_count: int = 0
    current_level: int = 0  # 已算出的最高（压缩最深）级别

    def char_count_at(self, level: int) -> int:
        """返回指定级别的字符数；尚未计算则返回 0。"""
        content = self.levels.get(level, "")
        return len(content)

### 压缩引擎

引擎负责真正的压缩。每次转换（L0 到 L1、L1 到 L2、L2 到 L3）都使用一条针对目标格式定制的 LLM 提示词。

- **L0 到 L1**：提取 3 到 5 个要点，输出为列表项。
- **L1 到 L2**：把这些要点蒸馏成一句话。
- **L2 到 L3**：只抽取实体名称和主题标签，输出为 JSON 对象。

每一步都是独立的 LLM 调用。这样提示词保持聚焦，输出也可预测。


In [ ]:
class CompactionEngine:
    """用 LLM 调用把记忆从上一级压缩到下一级。"""

    def __init__(self, model):
        self.model = model

    def _call_llm(self, system_prompt: str, user_content: str) -> str:
        """发送一次聊天补全请求。"""
        response = self.model.invoke([
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ])
        return to_text(response.content)

    def compact_to_l1(self, raw_text: str) -> str:
        """L0 -> L1：提取 3 到 5 个要点，输出为列表项。"""
        prompt = (
            "Extract the 3 to 5 most important points from this text. "
            "Return only bullet points, one per line, starting with '- '. "
            "Keep each point to one sentence."
        )
        return self._call_llm(prompt, raw_text)

    def compact_to_l2(self, key_points: str) -> str:
        """L1 -> L2：把要点蒸馏成一句话。"""
        prompt = (
            "Distill these key points into a single sentence of at most "
            "25 words. Return only the sentence, no extra text."
        )
        return self._call_llm(prompt, key_points)

    def compact_to_l3(self, summary: str) -> tuple:
        """L2 -> L3：抽取标签和实体，返回结构化数据。

        返回 (tag_string, entity_list)。
        """
        prompt = (
            "Extract entity tags from this text. Return a JSON object with "
            "two keys: \"tags\" (topic keywords as a list of strings) and "
            "\"entities\" (proper nouns like people, projects, tools as a "
            "list of strings). Return only valid JSON, no markdown fences."
        )
        raw = self._call_llm(prompt, summary)
        # 若模型用 markdown 代码围栏包住 JSON，先剥掉围栏
        cleaned = re.sub(r"^```(?:json)?\n?|```$", "", raw.strip())
        parsed = json.loads(cleaned)
        tag_str = ", ".join(parsed.get("tags", []))
        entities = parsed.get("entities", [])
        return tag_str, entities

`compact` 方法负责编排整个渐进压缩。它从记忆当前所在级别逐级走到目标级别，每一步调用对应的方法。例如从 L0 压缩到 L3，会依次执行 L0 到 L1、L1 到 L2、L2 到 L3。


In [ ]:
def _compact(self, memory: CompactedMemory, target_level: int) -> CompactedMemory:
    """把记忆逐级压缩到目标级别。

    需要时逐个执行中间步骤。例如从 L0 压缩到 L3，
    会依次执行 L0->L1、L1->L2、L2->L3。
    """
    current = memory.current_level
    while current < target_level:
        next_level = current + 1
        source_text = memory.levels[current]

        if next_level == CompactionLevel.L1_KEY_POINTS:
            memory.levels[next_level] = self.compact_to_l1(source_text)
        elif next_level == CompactionLevel.L2_SUMMARY:
            memory.levels[next_level] = self.compact_to_l2(source_text)
        elif next_level == CompactionLevel.L3_TAGS:
            tag_str, entities = self.compact_to_l3(source_text)
            memory.levels[next_level] = tag_str
            memory.entities = entities

        memory.current_level = next_level
        current = next_level

    return memory

CompactionEngine.compact = _compact


### 记忆存储

存储器保存所有记忆及其压缩级别。它还管理冷存储：当一条记忆被压缩超过 L0 后，原始文本会移入单独的归档。这样活跃存储保持精简，需要时仍能取回完整保真度的内容。


In [ ]:
class MemoryStore:
    """在多个压缩级别上存储记忆，并附带冷存储。"""

    def __init__(self, engine: CompactionEngine):
        self.engine = engine
        self.memories: dict[str, CompactedMemory] = {}
        self.cold_storage: dict[str, str] = {}  # memory_id -> L0 文本

    def add(self, raw_text: str, memory_id: str | None = None) -> CompactedMemory:
        """以 L0 级别新增一条原始记忆。"""
        if memory_id is None:
            memory_id = uuid.uuid4().hex[:8]
        now = time.time()
        memory = CompactedMemory(
            memory_id=memory_id,
            levels={CompactionLevel.L0_RAW: raw_text},
            entities=[],
            created_at=now,
            last_accessed=now,
        )
        self.memories[memory_id] = memory
        return memory

    def compact(self, memory_id: str, target_level: int) -> CompactedMemory:
        """把记忆压缩到目标级别，并将 L0 归档到冷存储。"""
        memory = self.memories[memory_id]
        self.engine.compact(memory, target_level)

        # 压缩发生后把 L0 移入冷存储
        if target_level >= CompactionLevel.L1_KEY_POINTS:
            if memory_id not in self.cold_storage:
                self.cold_storage[memory_id] = memory.levels[CompactionLevel.L0_RAW]

        return memory

`retrieve` 方法最能体现多级别访问的价值。你请求一个具体的详细级别：已计算好的级别立即返回；原始 L0 已归档的，从冷存储取回；还没计算的级别，存储器按需触发压缩。


In [ ]:
def _retrieve(self, memory_id: str, level: int | None = None) -> str:
    """按请求的详细级别检索记忆。

    如果该级别尚未计算，就按需触发压缩。
    如果请求 L0 且已被归档，则从冷存储读取。
    """
    memory = self.memories[memory_id]
    memory.last_accessed = time.time()
    memory.access_count += 1

    if level is None:
        level = memory.current_level

    # 检查该级别是否已经可用
    if level in memory.levels:
        return memory.levels[level]

    # L0 可能已被移入冷存储
    if level == CompactionLevel.L0_RAW and memory_id in self.cold_storage:
        return self.cold_storage[memory_id]

    # 请求的级别尚未计算：按需压缩
    if level > memory.current_level:
        self.compact(memory_id, level)
        return memory.levels[level]

    return memory.levels[memory.current_level]

def _storage_report(self) -> dict:
    """统计各压缩级别的字符数。"""
    totals = {level: 0 for level in CompactionLevel}
    for memory in self.memories.values():
        for level_key, content in memory.levels.items():
            totals[level_key] += len(content)
    cold_chars = sum(len(v) for v in self.cold_storage.values())
    return {
        "active_levels": {
            CompactionLevel(k).name: v for k, v in totals.items()
        },
        "cold_storage_chars": cold_chars,
        "total_memories": len(self.memories),
    }

MemoryStore.retrieve = _retrieve

MemoryStore.storage_report = _storage_report


### 压缩调度器

在生产系统中，你不会手动压缩每条记忆。调度器会检查哪些记忆需要继续压缩。它使用两种策略：

- **按存留时间**：压缩超过阈值（例如 24 小时）的记忆。
- **按空闲时间**：压缩近期未被访问的记忆。

你可以用同样的模式添加更多策略（存储压力、访问频率）。


In [ ]:
class CompactionScheduler:
    """按存留时间和访问模式挑选待压缩的记忆。"""

    def __init__(
        self,
        store: MemoryStore,
        age_threshold_seconds: float = 86400,   # 24 小时
        idle_threshold_seconds: float = 3600,    # 1 小时未访问
    ):
        self.store = store
        self.age_threshold = age_threshold_seconds
        self.idle_threshold = idle_threshold_seconds

    def get_candidates(self) -> list:
        """找出符合进一步压缩条件的记忆。

        返回 (memory_id, target_level) 元组的列表。
        """
        now = time.time()
        candidates = []
        for mid, memory in self.store.memories.items():
            # 已处于最高压缩级别
            if memory.current_level >= CompactionLevel.L3_TAGS:
                continue

            age = now - memory.created_at
            idle_time = now - memory.last_accessed
            next_level = memory.current_level + 1

            if age > self.age_threshold:
                candidates.append((mid, next_level))
            elif idle_time > self.idle_threshold:
                candidates.append((mid, next_level))

        return candidates

    def run(self) -> list:
        """压缩所有符合条件的记忆，返回已压缩的 ID 列表。"""
        candidates = self.get_candidates()
        compacted_ids = []
        for mid, target_level in candidates:
            self.store.compact(mid, target_level)
            compacted_ids.append(mid)
            print(f"  Compacted {mid} to {CompactionLevel(target_level).name}")
        return compacted_ids

## 运行示例

我们来完整走一遍场景：添加三条贴近实际的智能体记忆，把它们逐级压缩过全部四个级别，观察存储在每一步缩小了多少。


先创建引擎和存储器，然后添加三条智能体在一个工作日里可能收集到的原始记忆。


In [ ]:
engine = CompactionEngine(model=llm)
store = MemoryStore(engine=engine)

# 记忆 1：一次 CI/CD 规划对话
memory_1 = store.add(
    memory_id="cicd-plan",
    raw_text=(
        "User: We need to set up CI/CD for the new microservices project. "
        "We're using Kubernetes on AWS EKS. The team uses GitHub for version control. "
        "Assistant: I'll help you set up CI/CD for your Kubernetes microservices on "
        "AWS EKS. I recommend GitHub Actions for the pipeline. Each service gets its "
        "own workflow file. The pipeline should build a Docker image, push it to ECR, "
        "then deploy via Helm charts. User: Sounds good. What about staging vs "
        "production? Assistant: Use separate EKS namespaces for staging and production. "
        "The staging deploy triggers on every push to main. Production requires a "
        "manual approval step in the GitHub Actions workflow. User: What about "
        "rollbacks? Assistant: Helm tracks release history. Run 'helm rollback' to "
        "revert to any previous version. Set up health checks so Kubernetes "
        "auto-restarts unhealthy pods. User: Great, let's go with that plan."
    ),
)

# 记忆 2：一次调试会话
memory_2 = store.add(
    memory_id="debug-500",
    raw_text=(
        "User: The checkout service is throwing 500 errors since yesterday's deploy. "
        "About 15% of requests fail. Assistant: Let me help debug that. What changed "
        "in yesterday's deploy? User: We updated the payment gateway SDK from v2.3 to "
        "v3.0 and added a new discount calculation module. Assistant: The payment SDK "
        "jump from v2 to v3 is likely the cause. v3 changed the response format from "
        "XML to JSON. Your deserialization code probably expects XML. Check the "
        "PaymentResponseParser class. User: You're right. The parser was calling "
        "xml.etree.parse on JSON responses. I've switched it to json.loads and the "
        "errors stopped. Assistant: Good catch. I'd also recommend adding a response "
        "format check so future SDK updates don't break silently. Log the content-type "
        "header before parsing."
    ),
)

print(f"Stored {len(store.memories)} memories at L0.")


再添加第三条记忆：一次产品路线图讨论。三条都添加完后，我们查看它们在 L0 的字符数。


In [ ]:

# 记忆 3：一次产品路线图讨论
memory_3 = store.add(
    memory_id="q2-roadmap",
    raw_text=(
        "User: We need to plan the Q2 product roadmap. Main themes are mobile app "
        "improvements and API v2 launch. The team has 12 engineers. Assistant: Let's "
        "break Q2 into two tracks. Track A: Mobile improvements with 5 engineers. "
        "Track B: API v2 with 7 engineers. User: For mobile, the priorities are "
        "offline mode, push notifications, and biometric login. Assistant: I'd suggest "
        "this order: biometric login first (2 weeks, unblocks other features), then "
        "push notifications (3 weeks, high user demand), then offline mode (5 weeks, "
        "most complex). User: For API v2, we need rate limiting, webhook support, and "
        "OAuth2 scopes. Assistant: Start with OAuth2 scopes since other features "
        "depend on the auth model. Then rate limiting (protects the new endpoints). "
        "Webhooks last because they need the stable endpoint structure. User: Timeline "
        "works. Let's write this up and share with the team."
    ),
)

print(f"Added {len(store.memories)} memories to the store.")
for mid, mem in store.memories.items():
    chars = mem.char_count_at(CompactionLevel.L0_RAW)
    print(f"  {mid}: {chars} characters at L0")

### 逐级压缩

我们把第一条记忆一次压缩一级，看看 LLM 在每个阶段到底产出什么。


In [ ]:
# 把 cicd-plan 从 L0 压缩到 L1（要点）
store.compact("cicd-plan", CompactionLevel.L1_KEY_POINTS)
mem = store.memories["cicd-plan"]

print("=== L0: Raw transcript ===")
print(mem.levels[CompactionLevel.L0_RAW][:200] + "...")
print(f"\n({mem.char_count_at(CompactionLevel.L0_RAW)} chars)")

print("\n=== L1: Key points ===")
print(mem.levels[CompactionLevel.L1_KEY_POINTS])
print(f"\n({mem.char_count_at(CompactionLevel.L1_KEY_POINTS)} chars)")

ratio = mem.char_count_at(CompactionLevel.L1_KEY_POINTS) / mem.char_count_at(CompactionLevel.L0_RAW)
print(f"\nCompression ratio: {ratio:.0%} of original")

接着从 L1 压缩到 L2（单句摘要），再从 L2 压缩到 L3（只剩标签）。


In [ ]:
# 依次执行 L1 -> L2 -> L3
store.compact("cicd-plan", CompactionLevel.L3_TAGS)
mem = store.memories["cicd-plan"]

print("=== L2: One-sentence summary ===")
print(mem.levels[CompactionLevel.L2_SUMMARY])
print(f"({mem.char_count_at(CompactionLevel.L2_SUMMARY)} chars)")

print("\n=== L3: Tags only ===")
print(mem.levels[CompactionLevel.L3_TAGS])
print(f"({mem.char_count_at(CompactionLevel.L3_TAGS)} chars)")

print(f"\nExtracted entities: {mem.entities}")

# 展示完整的压缩阶梯
print("\n=== Compression ladder ===")
for level in CompactionLevel:
    if level in mem.levels:
        chars = len(mem.levels[level])
        pct = chars / mem.char_count_at(CompactionLevel.L0_RAW) * 100
        print(f"  {level.name}: {chars} chars ({pct:.1f}% of L0)")

### 批量压缩

把剩下的两条记忆各用一次调用压缩到 L3。引擎会自动执行全部中间步骤。


In [ ]:
for mid in ["debug-500", "q2-roadmap"]:
    store.compact(mid, CompactionLevel.L3_TAGS)
    mem = store.memories[mid]
    l0_chars = mem.char_count_at(CompactionLevel.L0_RAW)
    l3_chars = mem.char_count_at(CompactionLevel.L3_TAGS)
    print(f"{mid}:")
    print(f"  L0: {l0_chars} chars")
    print(f"  L3: {l3_chars} chars ({l3_chars / l0_chars:.1%} of L0)")
    print(f"  Entities: {mem.entities}")
    print()

### 按级别检索

这就是多级别存储的关键优势：你可以为每次查询选择合适的分辨率。宽泛的搜索扫 L3 标签，聚焦的查找读 L1 要点，完整的审计则从冷存储取回 L0。


In [ ]:
print("--- Retrieving 'debug-500' at different levels ---\n")

for level in CompactionLevel:
    content = store.retrieve("debug-500", level=level)
    preview = content[:120].replace("\n", " ")
    if len(content) > 120:
        preview += "..."
    print(f"{level.name}:")
    print(f"  {preview}")
    print(f"  ({len(content)} chars)\n")

### 存储节省

我们来测量压缩在三条记忆上总共省下多少空间。


In [ ]:
report = store.storage_report()

print(f"Total memories: {report['total_memories']}")
print(f"\nActive store by level:")
for level_name, chars in report["active_levels"].items():
    print(f"  {level_name}: {chars:,} chars")
print(f"\nCold storage (archived L0): {report['cold_storage_chars']:,} chars")

# 计算整体压缩率
total_l0 = sum(
    mem.char_count_at(CompactionLevel.L0_RAW)
    for mem in store.memories.values()
)
total_l3 = sum(
    mem.char_count_at(CompactionLevel.L3_TAGS)
    for mem in store.memories.values()
)
print(f"\nTotal L0 content: {total_l0:,} chars")
print(f"Total L3 content: {total_l3:,} chars")
print(f"Compression ratio (L3/L0): {total_l3 / total_l0:.1%}")
print(f"\nIf you retrieve at L3, you use ~{total_l3 / total_l0:.0%} of the tokens.")
print(f"The original text stays safe in cold storage for when you need it.")

### 定时压缩

在真实系统里，你不会手动触发压缩。调度器周期性运行，压缩符合策略条件的记忆。下面用一个较短的空闲阈值来演示。


In [ ]:
# 为调度演示新建一个干净的存储器
sched_engine = CompactionEngine(client=client, model=MODEL)
sched_store = MemoryStore(engine=sched_engine)

# 添加两条记忆
sched_store.add(
    memory_id="old-mem",
    raw_text="User asked about Python virtual environments. We discussed venv, "
    "conda, and poetry. Recommended poetry for dependency management in "
    "production projects. Covered lock files and reproducible builds.",
)

sched_store.add(
    memory_id="new-mem",
    raw_text="User wants to learn about async programming in Python. Discussed "
    "asyncio event loops, coroutines, and the await keyword. Recommended "
    "starting with simple HTTP requests using aiohttp.",
)

# 把第一条记忆的时间往前拨，让它看起来很旧
sched_store.memories["old-mem"].created_at -= 90000   # 约 25 小时前
sched_store.memories["old-mem"].last_accessed -= 90000

# 创建调度器，存留时间阈值设为 24 小时
scheduler = CompactionScheduler(
    store=sched_store,
    age_threshold_seconds=86400,   # 24 小时
    idle_threshold_seconds=3600,   # 1 小时
)

print("Before scheduling:")
for mid, mem in sched_store.memories.items():
    print(f"  {mid}: level={CompactionLevel(mem.current_level).name}")

print("\nRunning scheduler...")
compacted = scheduler.run()

print(f"\nCompacted {len(compacted)} memories.")
print("\nAfter scheduling:")
for mid, mem in sched_store.memories.items():
    print(f"  {mid}: level={CompactionLevel(mem.current_level).name}")

## 权衡

### 记忆压缩的适用场景

- **长期运行的智能体**：几天或几周内积累成千上万条原始记忆。压缩让记忆库保持可检索，同时不让存储成本失控。
- **token 受限的系统**：检索到的上下文必须塞进紧张的预算。L2 和 L3 摘要让你在一条提示词里装下更多记忆。
- **多分辨率检索**：先在 L3 上做基于标签的快速过滤，再把头部结果展开到 L1 获取更丰富的上下文。这种两段式模式既快又省。

### 什么时候会失效

- **对延迟敏感的写入。** 每一步压缩都要一次 LLM 调用。如果你要求写入即时完成，就得把压缩放进后台任务批量执行，这会增加运维复杂度。
- **需要精确回忆的任务。** 摘要是有损的。如果智能体之后需要第 47 轮的原话，L2 摘要里没有。冷存储检索可以补救，但会增加延迟。
- **压缩本身的成本。** 一条记忆到达 L3 最多需要 3 次 LLM 调用。对每小时产出数百条记忆的高吞吐智能体来说，压缩成本可能抵消存储上的节省。
- **小规模记忆库。** 如果智能体的记忆不到 100 条，多级别压缩的开销大于收益。缓冲记忆或窗口记忆是更好的选择。


## 延伸阅读

- Forte, ["Progressive Summarization: A Practical Technique for Designing Discoverable Notes,"](https://fortelabs.com/blog/progressive-summarization-a-practical-technique-for-designing-discoverable-notes/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) 2017。个人知识管理领域的渐进式摘要原始方法，本 notebook 的多级别压缩思路直接源于它。
- Xu et al., ["RECOMP: Improving Retrieval-Augmented LMs with Compression and Selective Augmentation,"](https://arxiv.org/abs/2310.04408) 2023。在把检索到的文档注入上下文之前先做压缩，证明压缩后的内容可以媲美甚至超过原始内容的表现。
- Chevalier et al., ["Adapting Language Models to Compress Contexts,"](https://arxiv.org/abs/2305.14788) 2023。训练模型生成紧凑的上下文表示，是一种学习式的记忆压缩方法。
- Jiang et al., ["LongLLMLingua: Accelerating and Enhancing LLMs in Long Context Scenarios via Prompt Compression,"](https://arxiv.org/abs/2310.06839) 2023。提示词压缩技术，在保持任务表现的同时减少 token 数量。


---

*← 上一篇：[14 - 记忆巩固](../14_memory_consolidation/) · 下一篇：[16 - 自我反思记忆](../16_self_reflection_memory/) →*


## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：自定义中间级别
在 `CompactionLevel` 的 L1（要点）和 L2（一句话摘要）之间新增一个 `L1_5_HIGHLIGHTS` 级别。在 `CompactionEngine` 里写一个 `compact_to_l1_5()` 方法，只提取最重要的 2 个要点。在 5 条记忆上测试它，并比较输出长度与 L1、L2 的差异。

### 挑战 2：压缩率分析
把 10 条记忆压缩过全部四个级别。对每条记忆，测量 L0、L1、L2、L3 各级的 token 数，计算每个级别的压缩率（L0 token 数 / LN token 数）。画一张柱状图，展示 10 条记忆的平均压缩率。

### 挑战 3：按级别检索
修改 `MemoryStore.retrieve()`，先搜索 L2 表示（快、省），再取回头部结果的 L0 或 L1 版本（细、准）。测量检索质量和 token 成本，与始终搜索 L0 做对比。这对应技术 20「记忆检索模式」中的混合检索思路。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--15-memory-compaction--memory-compaction)